# 1 · Select comparable controls
Run this notebook before notebook 2. The default is **Finca Rinconada (2022)**. All four user-supplied boundaries are saved in `examples/`; change `SITE_PATH` below to choose another, while continuing with one site at a time. The local exclusion file is empty at user request: no neighboring restoration projects are excluded in this example run. Add boundaries when available.

Matching uses pre-intervention AlphaEarth for interventions from 2018, otherwise the latest available pre-intervention GLAD land cover. Treatment samples characterize matching; they do not define the treated outcome.

In [24]:
from pathlib import Path

import sys
import json
from pathlib import Path

import ee
import pandas as pd
from IPython.display import display

import yaml

from restoration_counterfactuals.matching import (
    donor_search_area,
    matching_layers,
    sample_treatment_pixels,
    scale_by_sqrt_area,
    select_donors,
    similarity_masks,
)
from restoration_counterfactuals.plotting import matching_map

EE_PROJECT = "ee-speckerfelix"

if "google.colab" in sys.modules:
    from google.colab import output

    output.enable_custom_widget_manager()

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

ROOT = Path("..").resolve()
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

# # Local: launch from the repository or notebooks directory.
# # Colab: upload a zip of this repository, or set REPO_URL to its published Git URL.
# REPO_URL = ""  # No remote is configured in the source repository yet.
# if Path("pyproject.toml").exists():
#     ROOT = Path.cwd()
# elif Path("../pyproject.toml").exists():
#     ROOT = Path.cwd().parent
# elif Path("/content/restoration-counterfactuals/pyproject.toml").exists():
#     ROOT = Path("/content/restoration-counterfactuals")
# else:
#     if not REPO_URL:
#         raise ValueError("Upload/unzip the repository into /content/restoration-counterfactuals, or set REPO_URL.")
#     ROOT = Path("/content/restoration-counterfactuals")
#     subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)


In [25]:
# display the site

In [26]:
SITE_PATH = ROOT / "examples/reforest_now_au.geojson"  # Default: Finca Rinconada, 2022
site_json = json.loads(SITE_PATH.read_text())
if len(site_json["features"]) != 1:
    raise ValueError("Provide exactly one project boundary.")
site_feature = site_json["features"][0]
intervention_year = int(site_feature["properties"]["intervention_year"])
site = ee.Feature(site_feature)
exclusion_json = json.loads(
    (ROOT / "examples/exclusion_boundaries.geojson").read_text()
)
print(exclusion_json["coverage_note"])
exclusions = ee.FeatureCollection(exclusion_json["features"])
settings = yaml.safe_load((ROOT / "config.yaml").read_text())["CONTROL_MATCHING"]
area_ha = site.geometry().area(10).divide(10000).getInfo()
region = donor_search_area(site.geometry(), area_ha, exclusions, settings)
layers = matching_layers(intervention_year, region.bounds(10))
n_pixels = int(
    scale_by_sqrt_area(
        area_ha,
        settings["TREATMENT_PIXELS"]["N_MIN"],
        settings["TREATMENT_PIXELS"]["N_MAX"],
    )
)
print(
    {
        "area_ha": area_ha,
        "matching_mode": layers["mode"],
        "requested_treatment_pixels": n_pixels,
    }
)

Empty exclusion collection at user request; no neighboring restoration projects are excluded in this example run.
{'area_ha': 0.26540762165136017, 'matching_mode': 'alphaearth', 'requested_treatment_pixels': 5}


## Sample treatments and select donors

Donor eligibility is evaluated separately for each treatment sample. A donor must satisfy all of the following criteria:

- **Search region:** Within a **10–50 km buffer** around the project boundary, excluding a **0.5–5 km spillover buffer**. Both distances increase with the square root of project area, reaching their minimum at ≤1 ha and maximum at ≥1,000 ha.
- **Other restoration projects:** Exclude their boundaries plus a **50 m buffer**. The current example uses an empty exclusion collection, so no neighboring projects are excluded.
- **Human modification:** The absolute difference between donor and treatment human-modification scores must be **≤0.10**. Scores represent the year before intervention, interpolated between five-year observations and capped at 2020.
- **Disturbance history:** If the treatment pixel experienced pre-intervention forest loss, the donor must also have pre-intervention loss, with loss years differing by **≤1 year**. Otherwise, the donor must belong to the **“no pre-intervention loss”** category, which combines no recorded loss with loss occurring in or after the intervention year.
- **Ecological similarity:** For interventions from **2018 onward**, use the previous year’s AlphaEarth embeddings and require embedding distance (**1 − dot-product similarity**) to be **≤0.10**. For earlier interventions, require the **same remapped land-cover class** using the latest available pre-intervention GLAD snapshot.

Sample **5–30 treatment pixels**, with the requested count increasing with the square root of project area. Sampling uses a **20 m scale**, selecting up to **5 eligible donors per treatment pixel**. Duplicate donor locations are removed before fitting.

All settings can be changed in `config.yaml`. 

In [27]:
treatments = sample_treatment_pixels(site, layers, n_pixels, settings["SAMPLE_SCALE"])
treatment_json = treatments.getInfo()
for feature in treatment_json["features"]:
    feature["geometry"].pop("geodesic", None)
if not treatment_json["features"]:
    raise ValueError(
        "No valid treatment pixels; inspect boundary and matching-layer coverage."
    )
# Materialize samples once before donor selection.
treatments = ee.FeatureCollection(treatment_json)
donors = select_donors(treatments, region, layers, intervention_year, settings)
donor_json = donors.getInfo()
for feature in donor_json["features"]:
    feature["geometry"].pop("geodesic", None)
if not donor_json["features"]:
    raise ValueError(
        "No eligible donors; inspect masks and configuration before proceeding."
    )
donors = ee.FeatureCollection(donor_json)
print(f"Selected {len(donor_json['features'])} unique donor locations.")

Selected 25 unique donor locations.


In [28]:
donors, treatments

(<ee.featurecollection.FeatureCollection at 0x1202bdfa0>,
 <ee.featurecollection.FeatureCollection at 0x1202bd580>)

In [29]:
masks = similarity_masks(
    ee.Feature(treatments.first()), layers, intervention_year, settings
)
site_map = matching_map(site, region, treatments, donors, masks, exclusions)
display(site_map)

Map(center=[-17.252318152250666, 145.6539735146001], controls=(WidgetControl(options=['position', 'transparent…

Toggle the individual mask layers: these illustrate matching against the **first treatment sample**; selected donors come from all samples. Duplicate donor coordinates are collapsed before fitting so repetition cannot change the ridge penalty.

In [30]:
(OUTPUT / "treatment_locations.geojson").write_text(json.dumps(treatment_json))
(OUTPUT / "donor_locations.geojson").write_text(json.dumps(donor_json))
# Persist the exact inputs so notebook 2 cannot silently use a changed site/year.
(OUTPUT / "site.geojson").write_text(json.dumps(site_json))
(OUTPUT / "matching_inputs.json").write_text(
    json.dumps(
        {
            "settings": settings,
            "exclusions": exclusion_json,
            "matching_mode": layers["mode"],
        },
        indent=2,
    )
)
print(f"Saved notebook 2 inputs in {OUTPUT}")

Saved notebook 2 inputs in /Users/felix/Projects/restoration-counterfactuals/outputs


## Continue in notebook 2
Locally the outputs directory is shared. Separate Colab notebooks have separate runtimes: download `matching_outputs.zip`, then upload and extract it into the repository root in notebook 2 (creating `outputs/`).

In [31]:
from zipfile import ZipFile

archive = OUTPUT / "matching_outputs.zip"
with ZipFile(archive, "w") as bundle:
    for filename in (
        "treatment_locations.geojson",
        "donor_locations.geojson",
        "site.geojson",
        "matching_inputs.json",
    ):
        bundle.write(OUTPUT / filename, f"outputs/{filename}")
print(archive)
# In Colab, use the Files sidebar to download this archive.


/Users/felix/Projects/restoration-counterfactuals/outputs/matching_outputs.zip
